# all_predict 0.3.0 Walkthrough: Classification, Regression, CLI, Metrics, Tuning, and Model Saving

This notebook is a practical end-to-end walkthrough of `all_predict` 0.3.0. It uses only local datasets from scikit-learn and generated data.

In [ ]:
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer, load_diabetes, load_wine, make_regression
from sklearn.model_selection import train_test_split

import all_predict
from all_predict import AllClassifier, AllRegressor, load_model, save_model
from all_predict.plotting import (
    plot_classification_results,
    plot_metric_comparison,
    plot_regression_results,
    plot_time_vs_score,
)

OUTPUT_ROOT = Path('examples/generated')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('all_predict version:', all_predict.__version__)

## 1. Binary Classification

Use the breast cancer dataset for a high-signal binary classification baseline.

In [ ]:
bc_X, bc_y = load_breast_cancer(return_X_y=True, as_frame=True)
bc_X_train, bc_X_test, bc_y_train, bc_y_test = train_test_split(
    bc_X, bc_y, test_size=0.2, random_state=42, stratify=bc_y
)

clf = AllClassifier(
    verbose=False,
    random_state=42,
    n_jobs=-1,
    predictions=True,
    sort_by='roc_auc',
    include_models=[
        'LogisticRegression',
        'RandomForestClassifier',
        'ExtraTreesClassifier',
        'GradientBoostingClassifier',
        'RidgeClassifier',
        'KNeighborsClassifier',
        'SVC',
        'LinearSVC',
        'GaussianNB',
        'DummyClassifier',
    ],
    output_dir=OUTPUT_ROOT / 'binary_classification',
    save_best=True,
)

binary_results, binary_predictions = clf.fit(bc_X_train, bc_X_test, bc_y_train, bc_y_test)
binary_results.head(10)

In [ ]:
binary_results[[
    'Model',
    'Accuracy',
    'Balanced Accuracy',
    'ROC AUC',
    'Average Precision',
    'F1 Weighted',
    'MCC',
    'Train Time',
]].head(10)

In [ ]:
plot_classification_results(binary_results, metric='roc_auc', top_n=10)

In [ ]:
plot_time_vs_score(binary_results, score_metric='roc_auc')

## 2. Multiclass Classification

Use the wine dataset and keep ROC AUC handling safe. Some models can score multiclass ROC AUC through `predict_proba`, some cannot.

In [ ]:
wine_X, wine_y = load_wine(return_X_y=True, as_frame=True)
wine_X_train, wine_X_test, wine_y_train, wine_y_test = train_test_split(
    wine_X, wine_y, test_size=0.25, random_state=42, stratify=wine_y
)

multiclass_clf = AllClassifier(
    verbose=False,
    random_state=42,
    n_jobs=-1,
    sort_by='f1_weighted',
    include_models=[
        'LogisticRegression',
        'RandomForestClassifier',
        'KNeighborsClassifier',
        'GradientBoostingClassifier',
        'GaussianNB',
        'SVC',
        'LinearSVC',
        'DummyClassifier',
    ],
)

multiclass_results, _ = multiclass_clf.fit(wine_X_train, wine_X_test, wine_y_train, wine_y_test)
multiclass_results[[
    'Model', 'F1 Macro', 'F1 Weighted', 'Balanced Accuracy', 'ROC AUC', 'Notes'
]].head(10)

When `ROC AUC` is unavailable in multiclass mode, the package keeps the run alive and records the limitation in the `Notes` column instead of failing the full comparison.

## 3. Regression

Use the diabetes dataset for a realistic regression example.

In [ ]:
db_X, db_y = load_diabetes(return_X_y=True, as_frame=True)
db_X_train, db_X_test, db_y_train, db_y_test = train_test_split(db_X, db_y, test_size=0.2, random_state=42)

reg = AllRegressor(
    verbose=False,
    random_state=42,
    n_jobs=-1,
    predictions=True,
    sort_by='r2',
    include_models=[
        'LinearRegression',
        'Ridge',
        'RandomForestRegressor',
        'ExtraTreesRegressor',
        'GradientBoostingRegressor',
        'KNeighborsRegressor',
        'SVR',
        'LinearSVR',
        'MLPRegressor',
        'DummyRegressor',
    ],
    output_dir=OUTPUT_ROOT / 'regression',
    save_best=True,
)

regression_results, regression_predictions = reg.fit(db_X_train, db_X_test, db_y_train, db_y_test)
regression_results.head(10)

In [ ]:
regression_results[[
    'Model',
    'R2',
    'Adjusted R2',
    'RMSE',
    'MAE',
    'Median AE',
    'Explained Variance',
    'Train Time',
]].head(10)

In [ ]:
plot_regression_results(regression_results, metric='r2', top_n=10)

In [ ]:
plot_time_vs_score(regression_results, score_metric='r2')

## 4. High-Signal Synthetic Regression

This section is for package demonstration and testability, not for fake benchmark claims.

In [ ]:
syn_X, syn_y = make_regression(n_samples=300, n_features=12, noise=5.0, random_state=42)

synthetic_reg = AllRegressor(
    verbose=False,
    random_state=42,
    include_models=['LinearRegression', 'RandomForestRegressor', 'ExtraTreesRegressor', 'DummyRegressor'],
)

synthetic_results, _ = synthetic_reg.fit(syn_X, syn_y, test_size=0.25, random_state=42)
synthetic_results[['Model', 'R2', 'RMSE', 'MAE']].head(10)

## 5. Mixed-Type DataFrame Demo

Create a small DataFrame with numeric, categorical, boolean, and missing values to show the preprocessing pipeline working end to end.

In [ ]:
mixed_frame = pd.DataFrame({
    'age': [35, 41, 29, None, 50, 38, 44, 31, None, 53],
    'bmi': [22.1, None, 31.2, 27.5, 29.1, 24.7, 28.4, None, 30.2, 26.5],
    'city': ['Kolkata', 'Delhi', None, 'Mumbai', 'Delhi', 'Kolkata', 'Delhi', 'Mumbai', 'Kolkata', None],
    'segment': ['a', 'a', 'b', 'b', 'c', 'c', 'a', 'b', None, 'c'],
    'active': pd.Series([True, False, True, None, False, True, False, True, None, False], dtype='boolean'),
})
mixed_target = pd.Series([1, 0, 1, 0, 0, 1, 0, 1, 1, 0], name='target')

mixed_clf = AllClassifier(
    verbose=False,
    predictions=True,
    include_models=['LogisticRegression', 'RandomForestClassifier', 'DummyClassifier'],
)

mixed_results, mixed_predictions = mixed_clf.fit(mixed_frame, mixed_target, test_size=0.3, random_state=42)
mixed_results[['Model', 'Accuracy', 'ROC AUC']].head()

## 6. Optional Tuning

Tuning is slower and can overfit the chosen validation split. Keep it narrow and compare it against the untuned baseline.

In [ ]:
tuned_clf = AllClassifier(
    verbose=False,
    random_state=42,
    sort_by='roc_auc',
    include_models=['LogisticRegression', 'RandomForestClassifier', 'ExtraTreesClassifier', 'GradientBoostingClassifier'],
    tune=True,
    tune_top_n=3,
    tuner='randomized',
    cv=5,
)

untuned_results, _ = tuned_clf.fit(bc_X_train, bc_X_test, bc_y_train, bc_y_test)
untuned_results[['Model', 'ROC AUC', 'Train Time']].head()

In [ ]:
tuned_clf.tuned_results_

## 7. Model Persistence

Save the best model, load it again, and verify that predictions still match.

In [ ]:
persistence_path = OUTPUT_ROOT / 'manual_best_model.joblib'
save_model(clf.best_model_, persistence_path)
reloaded = load_model(persistence_path)
original_preds = clf.best_model_.predict(bc_X_test)
reloaded_preds = reloaded.predict(bc_X_test)
np.array_equal(original_preds, reloaded_preds)

## 8. CLI Demonstration

Write a local CSV and run the CLI through `subprocess`.

In [ ]:
breast_bundle = load_breast_cancer(as_frame=True)
cli_frame = breast_bundle.frame.copy()
cli_frame['target'] = breast_bundle.target
cli_csv = OUTPUT_ROOT / 'cli_demo_classification.csv'
cli_output = OUTPUT_ROOT / 'cli_demo_output'
cli_frame.to_csv(cli_csv, index=False)

command = [
    sys.executable,
    '-m',
    'all_predict.cli',
    'classify',
    '--file', str(cli_csv),
    '--target', 'target',
    '--output', str(cli_output),
    '--include-models', 'LogisticRegression,RandomForestClassifier,DummyClassifier',
    '--predictions',
    '--save-best',
]
completed = subprocess.run(command, capture_output=True, text=True, check=True)
print(completed.stdout)
sorted(path.name for path in cli_output.iterdir())

## 9. Interpreting Results

The top model in a quick benchmark is not automatically the production choice. You still need to think about leakage, external validation, calibration, drift, fairness, domain constraints, and deployment costs.

In [ ]:
plot_metric_comparison(binary_results, metrics=['Accuracy', 'Balanced Accuracy', 'ROC AUC', 'F1 Weighted'])

## 10. Final Summary

Use `all_predict` when you need a fast, honest baseline across common tabular models. Do not use it as a substitute for real model development, proper validation design, or domain review.